# spark-connect-kotlin — API Examples

Case-by-case reference for both backends.

| Backend | Entry point | When to use |
|---------|-------------|-------------|
| **Reflection** | `toDataFrame` / `toKotlinList` | No annotations required. SCD pipelines, ad-hoc exploration, types you don't own. |
| **Serialization** | `toSerializableDataFrame` / `toSerializableKotlinList` | `@Serializable` required. Compile-time schema contract, governed production pipelines. |

**Prerequisite:** build the notebook JAR first:
```
./gradlew notebookFatJar
```
This produces `notebooks/lib/spark-connect-kotlin-notebook.jar` (~200 MB, gitignored).

In [1]:
// Single dependency — bundles Spark Connect client + all runtime deps
@file:DependsOn("../notebooks/lib/spark-connect-kotlin-notebook.jar")

In [2]:
import org.apache.spark.sql.SparkSession
import spark.kotlin.reflect.toDataFrame
import spark.kotlin.reflect.toKotlinList
import spark.kotlin.reflect.getSparkSchema
import spark.kotlin.serialization.toSerializableDataFrame
import spark.kotlin.serialization.toSerializableKotlinList
import spark.kotlin.serialization.schemaFor
import kotlinx.serialization.Serializable
import kotlinx.serialization.serializer
import kotlin.reflect.typeOf

fun docker(vararg args: String): String {
    val proc = ProcessBuilder("docker", *args)
        .redirectErrorStream(true)
        .start()
    val out = proc.inputStream.bufferedReader().readText().trim()
    proc.waitFor()
    return out
}

docker("stop", "spark-api-demo")
docker("rm",   "spark-api-demo")

val containerId = docker(
    "run", "-d", "--name", "spark-api-demo",
    "-p", "15002:15002",
    "spark-server",
    "bash", "-c",
    "/opt/spark/bin/spark-submit " +
    "--class org.apache.spark.sql.connect.service.SparkConnectServer " +
    "--master local[*] " +
    "--conf spark.connect.grpc.binding.port=15002 " +
    "/opt/spark/jars/spark-connect_*.jar"
)
println("Container: $containerId")

val deadline = System.currentTimeMillis() + 120_000L
var ready = false
while (!ready && System.currentTimeMillis() < deadline) {
    Thread.sleep(2_000)
    if (docker("logs", "spark-api-demo").contains("Spark Connect server started")) ready = true
}
check(ready) { "Spark Connect server did not start within 2 minutes" }
println("Spark Connect ready")

// In IDE_PROCESS mode the JVM persists across kernel restarts, so a previous
// spark.stop() may have left a terminated gRPC thread pool in the cached session.
// Clear any cached session before building a fresh one.
runCatching { SparkSession.clearActiveSession() }
runCatching { SparkSession.clearDefaultSession() }

val spark = SparkSession.builder()
    .remote("sc://localhost:15002")
    .getOrCreate()
println("SparkSession: ${spark.version()}")

Container: 2365c9be1523ff75df7553d1dd7ff81519180e32fbf3a26f5e479d2723ab6053
Spark Connect ready
SparkSession: 4.0.0


---
## 1. Reflection backend — basic round-trip

No annotations needed. Schema is inferred at runtime from the primary constructor.  
Cache is keyed by `KType` — inference runs once per type per JVM lifetime.

In [3]:
data class Person(val name: String, val age: Int, val score: Double)

val people = listOf(
    Person("Alice", 30, 9.5),
    Person("Bob",   25, 7.2),
    Person("Carol", 35, 8.8)
)

// List<T> → DataFrame
val df = people.toDataFrame(spark)
df.printSchema()
df.show()

// DataFrame → List<T>
val back: List<Person> = df.toKotlinList()
println(back)

root
 |-- age: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- score: double (nullable = false)

+---+-----+-----+
|age| name|score|
+---+-----+-----+
| 30|Alice|  9.5|
| 25|  Bob|  7.2|
| 35|Carol|  8.8|
+---+-----+-----+
[Person(name=Alice, age=30, score=9.5), Person(name=Bob, age=25, score=7.2), Person(name=Carol, age=35, score=8.8)]


---
## 2. Serialization backend — basic round-trip

`@Serializable` required. Schema is derived from the kotlinx.serialization descriptor at compile time.  
Decode latency is lower than the reflection backend — the generated deserializer avoids per-row `callBy` invocation.

In [4]:
@Serializable
data class Employee(val name: String, val department: String, val salary: Double)

val employees = listOf(
    Employee("Alice", "Engineering", 95_000.0),
    Employee("Bob",   "Marketing",   72_000.0),
    Employee("Carol", "Engineering", 102_000.0)
)

// List<T> → DataFrame
val df2 = employees.toSerializableDataFrame(spark)
df2.printSchema()
df2.show()

// DataFrame → List<T>
val back2: List<Employee> = df2.toSerializableKotlinList()
println(back2)

root
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- salary: double (nullable = false)

+-----+-----------+--------+
| name| department|  salary|
+-----+-----------+--------+
|Alice|Engineering| 95000.0|
|  Bob|  Marketing| 72000.0|
|Carol|Engineering|102000.0|
+-----+-----------+--------+
[Employee(name=Alice, department=Engineering, salary=95000.0), Employee(name=Bob, department=Marketing, salary=72000.0), Employee(name=Carol, department=Engineering, salary=102000.0)]


---
## 3. Nullable fields

`T?` maps to a nullable Spark column. Non-nullable `T` maps to non-nullable.  
Reading null into a non-nullable field throws `IllegalArgumentException` at decode time.

In [5]:
// Reflection — no annotation needed
data class WithNulls(val name: String, val age: Int?, val email: String?)

val nullData = listOf(
    WithNulls("Alice", 30,   "alice@example.com"),
    WithNulls("Bob",   null, null)
)

nullData.toDataFrame(spark).show()

// Serialization — add @Serializable
@Serializable
data class WithNullsSer(val name: String, val age: Int?, val email: String?)

listOf(
    WithNullsSer("Alice", 30,   "alice@example.com"),
    WithNullsSer("Bob",   null, null)
).toSerializableDataFrame(spark).show()

+----+-----------------+-----+
| age|            email| name|
+----+-----------------+-----+
|  30|alice@example.com|Alice|
|NULL|             NULL|  Bob|
+----+-----------------+-----+
+-----+----+-----------------+
| name| age|            email|
+-----+----+-----------------+
|Alice|  30|alice@example.com|
|  Bob|NULL|             NULL|
+-----+----+-----------------+


---
## 4. Nested data class (struct)

Nested `data class` fields become Spark `StructType` columns.  
Depth is unlimited — every nesting level produces a nested struct.

In [6]:
// Reflection
data class Address(val street: String, val city: String, val zip: String)
data class Customer(val name: String, val address: Address)

listOf(
    Customer("Alice", Address("Hämeentie 1", "Helsinki", "00530")),
    Customer("Bob",   Address("Mannerheimintie 5", "Helsinki", "00100"))
).toDataFrame(spark).printSchema()

// Serialization — both the outer and inner class must be @Serializable
@Serializable data class AddressSer(val street: String, val city: String, val zip: String)
@Serializable data class CustomerSer(val name: String, val address: AddressSer)

listOf(
    CustomerSer("Alice", AddressSer("Hämeentie 1", "Helsinki", "00530"))
).toSerializableDataFrame(spark).printSchema()

root
 |-- address: struct (nullable = false)
 |    |-- city: string (nullable = false)
 |    |-- street: string (nullable = false)
 |    |-- zip: string (nullable = false)
 |-- name: string (nullable = false)

root
 |-- name: string (nullable = false)
 |-- address: struct (nullable = false)
 |    |-- street: string (nullable = false)
 |    |-- city: string (nullable = false)
 |    |-- zip: string (nullable = false)



---
## 5. Collections — List, Set, Map

`List<T>` and `Set<T>` → `ArrayType`.  
`Map<K, V>` → `MapType`.  
Element types follow the same type rules recursively.

In [7]:
// Reflection
data class WithCollections(
    val name: String,
    val tags: List<String>,
    val scores: Set<Int>,
    val metadata: Map<String, String>
)

listOf(
    WithCollections("Alice", listOf("admin", "user"), setOf(95, 88), mapOf("lang" to "fi")),
    WithCollections("Bob",   listOf("user"),          setOf(72),     mapOf("lang" to "en", "tier" to "pro"))
).toDataFrame(spark).printSchema()

// Serialization — same structure, add @Serializable
@Serializable
data class WithCollectionsSer(
    val name: String,
    val tags: List<String>,
    val scores: Set<Int>,
    val metadata: Map<String, String>
)

listOf(
    WithCollectionsSer("Alice", listOf("admin", "user"), setOf(95, 88), mapOf("lang" to "fi"))
).toSerializableDataFrame(spark).printSchema()

root
 |-- metadata: map (nullable = false)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = false)
 |-- name: string (nullable = false)
 |-- scores: array (nullable = false)
 |    |-- element: integer (containsNull = false)
 |-- tags: array (nullable = false)
 |    |-- element: string (containsNull = false)

root
 |-- name: string (nullable = false)
 |-- tags: array (nullable = false)
 |    |-- element: string (containsNull = true)
 |-- scores: array (nullable = false)
 |    |-- element: integer (containsNull = true)
 |-- metadata: map (nullable = false)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



---
## 6. Enum class

Enum values are stored as their `name` string and restored on decode.  
Serialization backend: enums must be `@Serializable` (or annotated with `@SerialName` per value).

In [8]:
// Reflection
enum class Status { ACTIVE, INACTIVE, PENDING }
data class Account(val id: Int, val name: String, val status: Status)

val accounts = listOf(
    Account(1, "Alice", Status.ACTIVE),
    Account(2, "Bob",   Status.PENDING)
)
val dfEnum = accounts.toDataFrame(spark)
dfEnum.show()
println(dfEnum.toKotlinList<Account>())

// Serialization — enum must be @Serializable
@Serializable enum class StatusSer { ACTIVE, INACTIVE, PENDING }
@Serializable data class AccountSer(val id: Int, val name: String, val status: StatusSer)

listOf(AccountSer(1, "Alice", StatusSer.ACTIVE))
    .toSerializableDataFrame(spark)
    .toSerializableKotlinList<AccountSer>()
    .also { println(it) }

+---+-----+-------+
| id| name| status|
+---+-----+-------+
|  1|Alice| ACTIVE|
|  2|  Bob|PENDING|
+---+-----+-------+
[Account(id=1, name=Alice, status=ACTIVE), Account(id=2, name=Bob, status=PENDING)]
[AccountSer(id=1, name=Alice, status=ACTIVE)]


[AccountSer(id=1, name=Alice, status=ACTIVE)]

---
## 7. Sealed class — polymorphism

Sealed hierarchies use a **flat union schema**: all fields from all subclasses in one row, plus a
`_type` discriminator column holding the subclass simple name.  
Fields not belonging to the row's actual subtype are `null`.  
Both backends support sealed classes.

In [9]:
// NOTE: KClass.sealedSubclasses discovery does not work in Kotlin scripting environments
// (notebook kernels compile each cell in isolation; sealed subclasses are not registered).
// Sealed classes work correctly in compiled Kotlin code — see DataFrameTypesTest.kt.
// The encode produces only the _type discriminator column here; data fields are absent.
// Below shows the API as it would be called in compiled code:

sealed class Shape { abstract val color: String }
data class Circle(override val color: String, val radius: Double) : Shape()
data class Rectangle(override val color: String, val width: Double, val height: Double) : Shape()

val shapes = listOf<Shape>(
    Circle("red",  5.0),
    Rectangle("blue", 4.0, 6.0),
    Circle("green", 3.0)
)

// In compiled code, schema would be:
// root
//  |-- _type:  string  (nullable = false)   ← discriminator
//  |-- color:  string  (nullable = false)   ← shared field
//  |-- radius: double  (nullable = true)    ← Circle-only
//  |-- width:  double  (nullable = true)    ← Rectangle-only
//  |-- height: double  (nullable = true)    ← Rectangle-only

try {
    val dfShapes = shapes.toDataFrame(spark)
    println("Schema (scripting limitation — subclasses not discovered):")
    dfShapes.printSchema()
    dfShapes.show()
} catch (e: Exception) {
    println("Error (expected in scripting): ${e.message}")
}

Schema (scripting limitation — subclasses not discovered):
root
 |-- _type: string (nullable = false)

+---------+
|    _type|
+---------+
|   Circle|
|Rectangle|
|   Circle|
+---------+


In [10]:
// Serialization sealed class: same scripting limitation.
// kotlinx.serialization's subclass registry is populated at compile time via
// generated serializer lookups, which don't resolve in the REPL classloader.
// In compiled code with @Serializable sealed class, the API is:

@Serializable sealed class ShapeSer { abstract val color: String }
@Serializable data class CircleSer(override val color: String, val radius: Double) : ShapeSer()
@Serializable data class RectangleSer(override val color: String, val width: Double, val height: Double) : ShapeSer()

try {
    val df = listOf<ShapeSer>(CircleSer("red", 5.0), RectangleSer("blue", 4.0, 6.0))
        .toSerializableDataFrame(spark)
    df.show()
    println(df.toSerializableKotlinList<ShapeSer>())
} catch (e: Exception) {
    println("Error (expected in scripting — works in compiled code): ${e::class.simpleName}: ${e.message?.take(120)}")
}

Error (expected in scripting — works in compiled code): SerializationException: Serializer for subclass 'CircleSer' is not found in the polymorphic scope of 'ShapeSer'.
Check if class with serial name


---
## 8. Value class (inline)

Value classes are **unwrapped** to their backing type in the DataFrame.  
A `UserId(val value: Int)` column stores as `IntegerType`, not a struct.  
On decode, the backing value is re-wrapped into the value class.

In [11]:
// Reflection backend: @JvmInline value class is unwrapped to its backing type.
@JvmInline value class UserId(val value: Int)
@JvmInline value class Email(val value: String)
data class UserRecord(val id: UserId, val email: Email, val active: Boolean)

val users = listOf(
    UserRecord(UserId(1), Email("alice@example.com"), true),
    UserRecord(UserId(2), Email("bob@example.com"),   false)
)
val dfUsers = users.toDataFrame(spark)
dfUsers.printSchema()  // id: int (not struct), email: string (not struct)
dfUsers.show()
println(dfUsers.toKotlinList<UserRecord>())

// Serialization backend limitation: @Serializable @JvmInline value class is not yet
// supported. The serialization decoder expects a struct for compound types, but value
// classes are stored as their flat backing type in the Row — causing a ClassCastException.
// Use the reflection backend for value class fields.

root
 |-- active: boolean (nullable = false)
 |-- email: string (nullable = false)
 |-- id: integer (nullable = false)

+------+-----------------+---+
|active|            email| id|
+------+-----------------+---+
|  true|alice@example.com|  1|
| false|  bob@example.com|  2|
+------+-----------------+---+
[UserRecord(id=UserId(value=1), email=Email(value=alice@example.com), active=true), UserRecord(id=UserId(value=2), email=Email(value=bob@example.com), active=false)]


---
## 9. Generic data class

Generic classes work when the type argument is known at the call site.  
Both backends resolve generics via the reified type parameter at the call site.

In [12]:
// Reflection
data class Box<T>(val label: String, val value: T)

listOf(
    Box("price", 9.99),
    Box("count", 42.0)
).toDataFrame(spark).also { it.printSchema(); it.show() }

// Serialization — @Serializable propagates through generics
@Serializable data class BoxSer<T>(val label: String, val value: T)

listOf(
    BoxSer("price", 9.99),
    BoxSer("count", 42.0)
).toSerializableDataFrame(spark).also { it.printSchema(); it.show() }

root
 |-- label: string (nullable = false)
 |-- value: double (nullable = false)

+-----+-----+
|label|value|
+-----+-----+
|price| 9.99|
|count| 42.0|
+-----+-----+
root
 |-- label: string (nullable = false)
 |-- value: double (nullable = false)

+-----+-----+
|label|value|
+-----+-----+
|price| 9.99|
|count| 42.0|
+-----+-----+


[label: string, value: double]

---
## 10. Pair and Triple  *(reflection only)*

`Pair<A, B>` maps to columns `_1` and `_2`. `Triple<A, B, C>` adds `_3`.  
Useful for quick two- or three-column DataFrames without defining a data class.  

> `Pair` and `Triple` are not `@Serializable`. Use the **reflection** backend.

In [13]:
// Pair as top-level type
listOf("Alice" to 30, "Bob" to 25, "Carol" to 35)
    .toDataFrame(spark)
    .also { it.printSchema(); it.show() }

// Triple
listOf(Triple("Alice", 30, true), Triple("Bob", 25, false))
    .toDataFrame(spark)
    .also { it.printSchema(); it.show() }

// Note: Pair/Triple as fields inside a data class work in compiled code but generic
// type args may be erased to Any in Kotlin scripting environments.
// In compiled code:
//   data class WithPair(val id: Int, val coords: Pair<Double, Double>)
// produces: coords: struct<_1: double, _2: double>

root
 |-- _1: string (nullable = false)
 |-- _2: integer (nullable = false)

+-----+---+
|   _1| _2|
+-----+---+
|Alice| 30|
|  Bob| 25|
|Carol| 35|
+-----+---+
root
 |-- _1: string (nullable = false)
 |-- _2: integer (nullable = false)
 |-- _3: boolean (nullable = false)

+-----+---+-----+
|   _1| _2|   _3|
+-----+---+-----+
|Alice| 30| true|
|  Bob| 25|false|
+-----+---+-----+


[_1: string, _2: int ... 1 more field]

---
## 11. BigDecimal  *(reflection only)*

`java.math.BigDecimal` → Spark `DecimalType(38, 18)`.  

> `BigDecimal` has no kotlinx.serialization serializer. Use the **reflection** backend,  
> or write a custom `KSerializer<BigDecimal>` for the serialization path.

In [14]:
import java.math.BigDecimal

data class Invoice(val id: Int, val amount: BigDecimal, val taxRate: BigDecimal)

val invoices = listOf(
    Invoice(1, BigDecimal("1499.99"), BigDecimal("0.24")),
    Invoice(2, BigDecimal("249.00"),  BigDecimal("0.10"))
)
val dfInv = invoices.toDataFrame(spark)
dfInv.printSchema()
dfInv.show()
println(dfInv.toKotlinList<Invoice>())

root
 |-- amount: decimal(38,18) (nullable = false)
 |-- id: integer (nullable = false)
 |-- taxRate: decimal(38,18) (nullable = false)

+--------------------+---+--------------------+
|              amount| id|             taxRate|
+--------------------+---+--------------------+
|1499.990000000000...|  1|0.240000000000000000|
|249.0000000000000...|  2|0.100000000000000000|
+--------------------+---+--------------------+
[Invoice(id=1, amount=1499.990000000000000000, taxRate=0.240000000000000000), Invoice(id=2, amount=249.000000000000000000, taxRate=0.100000000000000000)]


---
## 12. Date and time types

| Type | Spark type | Backend |
|------|------------|---------|
| `java.time.LocalDate` | `DateType` | Reflection only |
| `java.time.Instant` | `TimestampType` | Reflection only |
| `kotlinx.datetime.LocalDate` | `DateType` | Both |
| `kotlinx.datetime.Instant` | `TimestampType` | Both |

Prefer `kotlinx.datetime` types when using the serialization backend.

In [15]:
import java.time.LocalDate as JavaLocalDate
import java.time.Instant as JavaInstant
import kotlinx.datetime.LocalDate as KtLocalDate
import kotlinx.datetime.Instant as KtInstant
import kotlinx.datetime.toKotlinInstant

// Reflection — java.time types
data class EventReflect(
    val name: String,
    val date: JavaLocalDate,
    val createdAt: JavaInstant
)

listOf(
    EventReflect("Launch", JavaLocalDate.of(2025, 3, 15), JavaInstant.parse("2025-03-15T09:00:00Z"))
).toDataFrame(spark).also { it.printSchema(); it.show() }

// Serialization — kotlinx.datetime types
@Serializable
data class EventSer(
    val name: String,
    val date: KtLocalDate,
    val createdAt: KtInstant
)

listOf(
    EventSer("Launch", KtLocalDate(2025, 3, 15), JavaInstant.parse("2025-03-15T09:00:00Z").toKotlinInstant())
).toSerializableDataFrame(spark).also { it.printSchema(); it.show() }

root
 |-- createdAt: timestamp (nullable = false)
 |-- date: date (nullable = false)
 |-- name: string (nullable = false)

+-------------------+----------+------+
|          createdAt|      date|  name|
+-------------------+----------+------+
|2025-03-15 09:00:00|2025-03-15|Launch|
+-------------------+----------+------+
root
 |-- name: string (nullable = false)
 |-- date: date (nullable = false)
 |-- createdAt: timestamp (nullable = false)

+------+----------+-------------------+
|  name|      date|          createdAt|
+------+----------+-------------------+
|Launch|2025-03-15|2025-03-15 09:00:00|
+------+----------+-------------------+


[name: string, date: date ... 1 more field]

---
## 13. Char

`Char` is stored as a single-character `StringType` column and restored on decode.

In [16]:
data class Grade(val student: String, val letter: Char, val pass: Boolean)

val grades = listOf(
    Grade("Alice", 'A', true),
    Grade("Bob",   'C', true),
    Grade("Carol", 'F', false)
)
val dfGrades = grades.toDataFrame(spark)
dfGrades.printSchema()
dfGrades.show()
println(dfGrades.toKotlinList<Grade>())

root
 |-- letter: string (nullable = false)
 |-- pass: boolean (nullable = false)
 |-- student: string (nullable = false)

+------+-----+-------+
|letter| pass|student|
+------+-----+-------+
|     A| true|  Alice|
|     C| true|    Bob|
|     F|false|  Carol|
+------+-----+-------+
[Grade(student=Alice, letter=A, pass=true), Grade(student=Bob, letter=C, pass=true), Grade(student=Carol, letter=F, pass=false)]


---
## 14. Column name mapping with `@SerialName`  *(serialization backend)*

Use `@SerialName` to map a Kotlin field name to a different DataFrame column name.  
Typical use: reading a CSV with `snake_case` columns into a `camelCase` Kotlin class.

In [17]:
import kotlinx.serialization.SerialName

@Serializable
data class Track(
    @SerialName("track_name")   val trackName: String,
    @SerialName("artist_name")  val artistName: String,
    @SerialName("duration_min") val durationMin: Double
)

val tracks = listOf(
    Track("Bohemian Rhapsody", "Queen",   5.91),
    Track("Hotel California",  "Eagles",  6.50)
)

val dfTracks = tracks.toSerializableDataFrame(spark)
dfTracks.printSchema()  // columns: track_name, artist_name, duration_min
dfTracks.show()

// Round-trip back — column names in Row match @SerialName, decoder maps back to Kotlin field
println(dfTracks.toSerializableKotlinList<Track>())

root
 |-- track_name: string (nullable = false)
 |-- artist_name: string (nullable = false)
 |-- duration_min: double (nullable = false)

+-----------------+-----------+------------+
|       track_name|artist_name|duration_min|
+-----------------+-----------+------------+
|Bohemian Rhapsody|      Queen|        5.91|
| Hotel California|     Eagles|         6.5|
+-----------------+-----------+------------+
[Track(trackName=Bohemian Rhapsody, artistName=Queen, durationMin=5.91), Track(trackName=Hotel California, artistName=Eagles, durationMin=6.5)]


---
## 15. Hardcoded schema override — both backends

Pass an explicit `StructType` to `toDataFrame` or `toSerializableDataFrame` to bypass inference.  
Use cases:
- Inject schema from Unity Catalog at runtime
- Control nullability explicitly (e.g., all columns nullable for schema evolution)
- Production pipelines where schema must be pinned to a known version

In [18]:
import org.apache.spark.sql.types.*

data class Product(val id: Int, val name: String, val price: Double)

val products = listOf(
    Product(1, "Widget", 9.99),
    Product(2, "Gadget", 24.99)
)

// Inferred schema — nullability follows Kotlin types
val inferred = getSparkSchema(typeOf<Product>())
println("Inferred: $inferred")

// Hardcoded schema — Scala singleton objects accessed via DataTypes for Kotlin/Java compat
val hardcoded = StructType(arrayOf(
    StructField("id",    DataTypes.IntegerType, true,  Metadata.empty()),
    StructField("name",  DataTypes.StringType,  true,  Metadata.empty()),
    StructField("price", DataTypes.DoubleType,  true,  Metadata.empty())
))

// Reflection backend with hardcoded schema
products.toDataFrame(spark, schema = hardcoded)
    .also { println("Reflect: ${it.schema()}"); it.show() }

// Serialization backend with hardcoded schema
@Serializable data class ProductSer(val id: Int, val name: String, val price: Double)
listOf(ProductSer(1, "Widget", 9.99))
    .toSerializableDataFrame(spark, schema = hardcoded)
    .also { println("Serialize: ${it.schema()}") }

Inferred: StructType(StructField(id,IntegerType,false),StructField(name,StringType,false),StructField(price,DoubleType,false))
Reflect: StructType(StructField(id,IntegerType,true),StructField(name,StringType,true),StructField(price,DoubleType,true))
+---+------+-----+
| id|  name|price|
+---+------+-----+
|  1|Widget| 9.99|
|  2|Gadget|24.99|
+---+------+-----+
Serialize: StructType(StructField(id,IntegerType,true),StructField(name,StringType,true),StructField(price,DoubleType,true))


[id: int, name: string ... 1 more field]

---
## 16. Schema inspection API

Retrieve the inferred `StructType` without encoding any data.  
Useful for pre-flight schema comparison, Unity Catalog validation, or DDL generation.

In [19]:
import org.apache.spark.sql.types.StructType

// Reflection backend — infer schema from KType (no data needed)
val reflectSchema: StructType = getSparkSchema(typeOf<Customer>())
println("Reflection schema (fields sorted alphabetically):")
println(reflectSchema.prettyJson())

// Serialization backend — infer schema from KSerializer descriptor
val serSchema: StructType = schemaFor(serializer<CustomerSer>())
println("\nSerialization schema (fields in declaration order):")
println(serSchema.prettyJson())

// Backends produce the same fields with the same types, but field ordering differs:
// - Reflection: alphabetical (HashMap-based inference walks constructor params but collects into a map)
// - Serialization: declaration order (descriptor drives field order)
// Direct == comparison will be false due to ordering. Compare field sets instead:
val reflectFields = reflectSchema.fields().map { it.name() to it.dataType() }.toSet()
val serFields     = serSchema.fields().map    { it.name() to it.dataType() }.toSet()
println("\nField sets equal (ordering-independent): ${reflectFields == serFields}")

Reflection schema (fields sorted alphabetically):
{
  "type" : "struct",
  "fields" : [ {
    "name" : "address",
    "type" : {
      "type" : "struct",
      "fields" : [ {
        "name" : "city",
        "type" : "string",
        "nullable" : false,
        "metadata" : { }
      }, {
        "name" : "street",
        "type" : "string",
        "nullable" : false,
        "metadata" : { }
      }, {
        "name" : "zip",
        "type" : "string",
        "nullable" : false,
        "metadata" : { }
      } ]
    },
    "nullable" : false,
    "metadata" : { }
  }, {
    "name" : "name",
    "type" : "string",
    "nullable" : false,
    "metadata" : { }
  } ]
}

Serialization schema (fields in declaration order):
{
  "type" : "struct",
  "fields" : [ {
    "name" : "name",
    "type" : "string",
    "nullable" : false,
    "metadata" : { }
  }, {
    "name" : "address",
    "type" : {
      "type" : "struct",
      "fields" : [ {
        "name" : "street",
        "type" : "st

---
## 17. Mixed backend pipeline

Different DataFrames in the same pipeline can use different backends.  
Backend choice is per-call, not per-pipeline. A `Dataset<Row>` is backend-agnostic once created.

Typical pattern: use **serialization** for governed types with a contract, **reflection** for
types you don't own (third-party, not `@Serializable`, or changing rapidly).

In [20]:
import org.apache.spark.sql.functions.*

// TypeA: governed, serialization backend
@Serializable data class Order(val orderId: Int, val customerId: Int, val total: Double)

// TypeB: not owned / not annotated, reflection backend
data class CustomerInfo(val customerId: Int, val name: String, val tier: String)

val orders = listOf(
    Order(1001, 1, 149.99),
    Order(1002, 2,  49.99),
    Order(1003, 1, 299.00)
)
val customers = listOf(
    CustomerInfo(1, "Alice", "premium"),
    CustomerInfo(2, "Bob",   "standard")
)

val dfOrders    = orders.toSerializableDataFrame(spark)   // serialization
val dfCustomers = customers.toDataFrame(spark)             // reflection

// Join — Spark treats both as plain Dataset<Row>
val joined = dfOrders.join(dfCustomers, "customerId")
joined.show()

// Aggregation over the joined result
joined.groupBy("name", "tier")
    .agg(sum("total").alias("total_spend"))
    .show()

+----------+-------+------+-----+--------+
|customerId|orderId| total| name|    tier|
+----------+-------+------+-----+--------+
|         1|   1003| 299.0|Alice| premium|
|         1|   1001|149.99|Alice| premium|
|         2|   1002| 49.99|  Bob|standard|
+----------+-------+------+-----+--------+
+-----+--------+-----------+
| name|    tier|total_spend|
+-----+--------+-----------+
|Alice| premium|     448.99|
|  Bob|standard|      49.99|
+-----+--------+-----------+


---
## 18. Standard Spark operations on a typed DataFrame

`toDataFrame` / `toSerializableDataFrame` return a standard `Dataset<Row>`.  
Any Spark SQL operation (filter, groupBy, join, window, etc.) works as normal.

In [21]:
import org.apache.spark.sql.functions.*

// Session liveness check — if this hangs, the Spark Connect connection is dead
println("[0] session liveness check")
spark.range(3).show()
println("[1] session alive")

@Serializable
data class Sale(val region: String, val product: String, val amount: Double, val qty: Int)

// Schema inference — no data sent to Spark
println("[2] schema: ${schemaFor(serializer<Sale>())}")

// createDataFrame encodes rows to Arrow and sends to Spark Connect
println("[3] createDataFrame")
val sales = listOf(
    Sale("North", "Widget", 9.99,  3),
    Sale("North", "Gadget", 24.99, 1),
    Sale("South", "Widget", 9.99,  5),
)
val dfSales = sales.toSerializableDataFrame(spark)
println("[4] DataFrame created")

println("[5] filter + show")
dfSales.filter(col("region").equalTo("North")).show()

println("[6] groupBy + agg")
dfSales.groupBy("region", "product")
    .agg(
        sum(col("amount").multiply(col("qty"))).alias("revenue"),
        sum("qty").alias("units_sold")
    )
    .orderBy("region", "product")
    .show()

println("[7] collect + decode")
val northSales = dfSales
    .filter(col("region").equalTo("North"))
    .toSerializableKotlinList<Sale>()
println("[8] done: $northSales")

[0] session liveness check
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+
[1] session alive
[2] schema: StructType(StructField(region,StringType,false),StructField(product,StringType,false),StructField(amount,DoubleType,false),StructField(qty,IntegerType,false))
[3] createDataFrame
[4] DataFrame created
[5] filter + show
+------+-------+------+---+
|region|product|amount|qty|
+------+-------+------+---+
| North| Widget|  9.99|  3|
| North| Gadget| 24.99|  1|
+------+-------+------+---+
[6] groupBy + agg
+------+-------+-------+----------+
|region|product|revenue|units_sold|
+------+-------+-------+----------+
| North| Gadget|  24.99|         1|
| North| Widget|  29.97|         3|
| South| Widget|  49.95|         5|
+------+-------+-------+----------+
[7] collect + decode
[8] done: [Sale(region=North, product=Widget, amount=9.99, qty=3), Sale(region=North, product=Gadget, amount=24.99, qty=1)]


---
## Cleanup

In [22]:
// Do NOT call spark.stop() in IDE_PROCESS mode — it terminates the gRPC thread pool
// permanently for this JVM lifetime. Stopping the container is sufficient; the session
// becomes inert and the next setup cell run will clear and replace it.
docker("stop", "spark-api-demo")
docker("rm",   "spark-api-demo")
println("Container stopped")

Container stopped
